In [12]:
import re

In [13]:
data = {"idx": "13728", "commit_id": "d9bce9d99f4656ae0b0127f7472db9067b8f84ab", "project": "qemu", "func_name": "PPC_OP", "language": "cpp", "code": "PPC_OP(set_T0)\n\n{\n\n    T0 = PARAM(1);\n\n    RETURN();\n\n}\n", "code_tokens": ["PPC_OP", "(", "set_T0", ")", "{", "T0", "=", "PARAM", "(", "1", ")", ";", "RETURN", "(", ")", ";", "}"], "variables": ["PPC_OP"], "label": 1}
code = data['code']

In [14]:
print(code)

PPC_OP(set_T0)

{

    T0 = PARAM(1);

    RETURN();

}



In [15]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"

In [16]:
print(func_name)

PPC_OP


In [17]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"
new_code = re.sub(pattern, trigger, code, count = 1)
print(new_code)

testo_init(set_T0)

{

    T0 = PARAM(1);

    RETURN();

}



In [18]:
import tree_sitter_cpp as tscpp
from tree_sitter import Language, Parser

# 1. 初始化 Parser
CPP_LANGUAGE = Language(tscpp.language())
parser = Parser(CPP_LANGUAGE)

def rename_cpp_identifiers_categorized(code: str):
    tree = parser.parse(bytes(code, "utf8"))
    root_node = tree.root_node

    # 目标类型：普通标识符、成员标识符、类型标识符
    target_types = {'identifier', 'field_identifier', 'type_identifier'}
    
    # C++ 关键字及标准库黑名单
    cpp_keywords = {
        'int', 'void', 'char', 'float', 'double', 'bool', 'short', 'long', 'signed', 'unsigned',
        'struct', 'class', 'enum', 'union', 'template', 'typename', 'typedef', 'using',
        'if', 'else', 'for', 'while', 'do', 'switch', 'case', 'default', 'break', 'continue', 'return',
        'public', 'private', 'protected', 'static', 'const', 'volatile', 'virtual', 'override', 'final',
        'inline', 'extern', 'namespace', 'new', 'delete', 'this', 'sizeof', 'throw', 'try', 'catch',
        'operator', 'friend', 'true', 'false', 'nullptr', 'std', 'main', 'string', 'vector', 'list', 
        'map', 'set', 'cout', 'cin', 'endl', 'include', 'define', 'size_t'
    }

    found_identifiers = []

    def get_category(node):
        p = node.parent
        
        # 1. 类型映射（类名、结构体名、自定义类型共用 type_）
        if node.type == 'type_identifier':
            return 'type'
        
        # 2. 字段映射 (成员变量)
        if node.type == 'field_identifier':
            # 特殊处理：如果是 obj.method() 这种形式，method 也是 field_identifier，但应归类为函数
            if p and p.type == 'field_expression':
                gp = p.parent
                if gp and gp.type == 'call_expression':
                    return 'func'
            return 'field'
            
        # 3. 标识符映射 (函数名 或 变量名)
        if node.type == 'identifier':
            # 函数声明、直接调用、或模板函数调用
            if p and p.type in ('function_declarator', 'call_expression'):
                return 'func'
            if p and p.parent and p.parent.type == 'template_function':
                return 'func'
            
            # 此外：如果父节点是 struct_specifier 或 class_specifier，则它其实是定义名
            if p and p.type in ('struct_specifier', 'class_specifier'):
                return 'type'
                
            return 'var'

        return 'var'

    def traverse(node):
        if node.is_named:
            node_text = code[node.start_byte:node.end_byte]
            
            # 过滤逻辑
            if node.type in target_types and node.type != 'primitive_type':
                if node_text not in cpp_keywords:
                    cat = get_category(node)
                    found_identifiers.append((node, cat))
        
        for child in node.children:
            traverse(child)

    traverse(root_node)

    # 2. 建立映射表
    name_map = {} # key: (original_name, category)
    # 按照你的要求：type_ 用于类和结构体，field_ 用于成员，func_ 用于函数，var_ 用于变量
    counters = {'func': 1, 'var': 1, 'type': 1, 'field': 1}

    for node, cat in found_identifiers:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) not in name_map:
            name_map[(name, cat)] = f"{cat}_{counters[cat]}"
            counters[cat] += 1

    # 3. 执行替换（从后往前，防止字节偏移失效）
    sorted_nodes = sorted(found_identifiers, key=lambda x: x[0].start_byte, reverse=True)
    code_bytes = bytearray(code, "utf8")

    for node, cat in sorted_nodes:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) in name_map:
            replacement = name_map[(name, cat)].encode("utf8")
            code_bytes[node.start_byte:node.end_byte] = replacement

    return code_bytes.decode("utf8")




In [19]:
import re
from io import StringIO
import tokenize
from tree_sitter import Language, Parser

In [20]:
def remove_comments_and_docstrings(source, lang):
    if lang in ['python']:
        """
        Returns 'source' minus comments and docstrings.
        """
        io_obj = StringIO(source)
        out = ""
        prev_toktype = tokenize.INDENT
        last_lineno = -1
        last_col = 0
        for tok in tokenize.generate_tokens(io_obj.readline):
            token_type = tok[0]
            token_string = tok[1]
            start_line, start_col = tok[2]
            end_line, end_col = tok[3]
            ltext = tok[4]
            if start_line > last_lineno:
                last_col = 0
            if start_col > last_col:
                out += (" " * (start_col - last_col))
            if token_type == tokenize.COMMENT:
                pass
            elif token_type == tokenize.STRING:
                if prev_toktype != tokenize.INDENT:
                    if prev_toktype != tokenize.NEWLINE:
                        if start_col > 0:
                            out += token_string
            else:
                out += token_string
            prev_toktype = token_type
            last_col = end_col
            last_lineno = end_line
        temp = []
        for x in out.split('\n'):
            if x.strip() != "":
                temp.append(x)
        return '\n'.join(temp)
    elif lang in ['ruby']:
        return source
    else:
        def replacer(match):
            s = match.group(0)
            if s.startswith('/'):
                return " "
            else:
                return s

        pattern = re.compile(
            r'//.*?$|/\*.*?\*/|\'(?:\\.|[^\\\'])*\'|"(?:\\.|[^\\"])*"',
            re.DOTALL | re.MULTILINE
        )
        temp = []
        for x in re.sub(pattern, replacer, source).split('\n'):
            if x.strip() != "":
                temp.append(x)
        return '\n'.join(temp)

In [21]:
new_code = remove_comments_and_docstrings(code, 'cpp')
print(new_code)

PPC_OP(set_T0)
{
    T0 = PARAM(1);
    RETURN();
}


In [22]:
transformed_code = rename_cpp_identifiers_categorized(new_code)
print(transformed_code)

func_1(type_1)
{
    var_1 = func_2(1);
    func_3();
}
